In [ ]:
#Note - before running all cells at once read this - there are cells in which i am taking input from the user for pdf 
# and a question which user wants to ask. so if you run all cell at once, the interpreter will proceed without taking input so 
# run all the cells one by one to get desired output.


In [ ]:
!pip install -q \
langchain==1.0.1 \
langchain-core \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
faiss-cpu \
rank-bm25 \
pypdf \
ipywidgets \
transformers==4.57.1 \
tokenizers==0.22.1 \
sentence-transformers \
accelerate \
torch

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import pipeline
import ipywidgets as widgets
from IPython.display import display
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

In [ ]:
import os
import shutil
import ipywidgets as widgets
from IPython.display import display
defpdf = "./CEI_User_Guide.pdf"
up = widgets.FileUpload(accept=".pdf", multiple=False)
bt = widgets.Button(description="Process PDF")
op = widgets.Output()
def on_button_click(b):
    with op:
        op.clear_output()
        if up.value:
            upfile = next(iter(up.value.values()))
            with open("temp.pdf", "wb") as f:
                f.write(upfile["content"])
            print("PDF Uploaded")
        elif os.path.exists(defpdf):
            shutil.copy(defpdf, "temp.pdf")
            print("No PDF uploaded.")
            print("Using default PDF - temp.pdf")
        else:
            print("Either PDF not uploaded or default PDF not found.")
            return
bt.on_click(on_button_click)
display(
    widgets.HTML("<b>Upload a PDF (Optional) or Click Process PDF to use Default PDF</b>"),
    up,
    bt,
    op
)

In [ ]:
loader = PyPDFLoader('temp.pdf')
docs = loader.load()
textsplitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
chunks = textsplitter.split_documents(docs)

In [ ]:
embeddingmodels = ["sentence-transformers/all-MiniLM-L6-v2", "sentence-transformers/all-mpnet-base-v2"]
vector = []
for m in embeddingmodels:
    embeddings = HuggingFaceEmbeddings(model_name=m)
    v = FAISS.from_documents(chunks, embeddings)
    vector.append(v)

In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_new_tokens=256,
)
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
template = '''Answer the question based only on the following context:
{context}
Question: {question}'''
prompt = ChatPromptTemplate.from_template(template)
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

In [ ]:
default_question = "What information is available on the CEI Dashboard?"
question = input(
    "Enter your question (Press Enter to use a default question): "
).strip()
if not question:
    question = default_question
    print(f"\nUsing default question:\n{question}")

In [ ]:
print("\nPrinting Chunks")
print("\nThere are total two embedding models i used for embedding purpose and now i am printing all the chunks\n")
for i, vs in enumerate(vector):
    print(f"\nEmbedding Model: {embeddingmodels[i]}")
    retriever = vs.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(question)
    for j, doc in enumerate(docs):
        print(f"Chunk {j+1}: {doc.page_content[:200]}...")

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from langchain_community.retrievers import BM25Retriever
bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 3
vector_retriever = vs.as_retriever(search_kwargs={"k": 3})

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [ ]:
def rerank_documents(question, docs, top_k=3):
    pairs = [[question, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )
    return [doc for score, doc in ranked[:top_k]]

In [ ]:
print("\nQuestion:")
print(question)
for i, vs in enumerate(vector):
    print(f"\nEmbedding Model: {embeddingmodels[i]}")
    retriever = vs.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 8,
            "fetch_k": 20,
            "lambda_mult": 0.5
        }
    )
    docs = retriever.invoke(question)
    docs = rerank_documents(question, docs, top_k=3)
    context = format_docs(docs)
    result = (
        prompt
        | llm
    ).invoke(
        {
            "context": context,
            "question": question
        }
    )
    print("Answer:")
    print(result)